# Лабораторная работа №3
## Деревья решений в задачах классификации и регрессии. ROC-кривая

**Датасет**: `Spaceship Titanic` (подготовлен в ЛР1)

---

### Задачи:
1. Загрузить и обработать данные
2. **Регрессия**: предсказание `Age`
3. **Классификация**: предсказание `Transported` + ROC-кривая


---
## 1. Загрузка и предобработка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Загрузка
data = pd.read_csv('data.csv')

# --- Заполнение пропусков ---
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    data[col] = data[col].fillna(data[col].mean())

cat_cols = ['HomePlanet', 'Cabin', 'Destination']
for col in cat_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

bool_cols = ['CryoSleep', 'VIP']
for col in bool_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# --- Нормализация ---
for col in num_cols:
    min_val, max_val = data[col].min(), data[col].max()
    if max_val != min_val:
        data[col] = (data[col] - min_val) / (max_val - min_val)

# --- Кодирование ---
data['CryoSleep'] = data['CryoSleep'].astype(int)
data['VIP'] = data['VIP'].astype(int)
data_encoded = pd.get_dummies(data, columns=['HomePlanet', 'Destination'], drop_first=True)

# --- Формирование X и y ---
exclude_cols = ['PassengerId', 'Cabin', 'Name']
X = data_encoded.drop(columns=[col for col in exclude_cols + ['Transported'] if col in data_encoded.columns])
y = data_encoded['Transported']

# Разделение
train_idx = X.sample(frac=0.7, random_state=42).index
test_idx = X.drop(train_idx).index

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print(f"Обучающая выборка: {X_train.shape}")
print(f"Тестовая выборка: {X_test.shape}")

---
## 2. Регрессия: Предсказание `Age`

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Данные для регрессии
X_reg_train = X_train.drop(columns=['Age'])
X_reg_test = X_test.drop(columns=['Age'])
y_reg_train = data_encoded.loc[train_idx, 'Age']
y_reg_test = data_encoded.loc[test_idx, 'Age']

# Модель
regressor = DecisionTreeRegressor(
    max_depth=5,
    min_samples_leaf=10,
    random_state=42
)
regressor.fit(X_reg_train, y_reg_train)

# Предсказание
y_pred_reg = regressor.predict(X_reg_test)

# Метрики
mae = mean_absolute_error(y_reg_test, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_reg))

print(f"Регрессия: предсказание Age")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")

# Визуализация дерева
plt.figure(figsize=(20, 10))
plot_tree(regressor, feature_names=X_reg_train.columns, filled=True, rounded=True, fontsize=9)
plt.title("Дерево решений: Регрессия (Age)")
plt.show()

---
## 3. Классификация: Предсказание `Transported` + ROC-кривая

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, auc

# Убираем Age из признаков
X_cls_train = X_train.drop(columns=['Age'])
X_cls_test = X_test.drop(columns=['Age'])

# Модель
classifier = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=5,
    criterion='gini',
    random_state=42
)
classifier.fit(X_cls_train, y_train)

# Вероятности
y_proba = classifier.predict_proba(X_cls_test)[:, 1]

# ROC-кривая
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC-кривая (AUC = {roc_auc:.3f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-кривая: Классификация Transported')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

print(f"ROC-AUC: {roc_auc:.3f}")

# Визуализация дерева
plt.figure(figsize=(22, 12))
plot_tree(classifier, feature_names=X_cls_train.columns,
          class_names=['Не транспортирован', 'Транспортирован'],
          filled=True, fontsize=9, rounded=True)
plt.title("Дерево решений: Классификация (Transported)")
plt.show()

---
## Итог

| Задача          | Цель         | Метрика     | Значение  |
|-----------------|--------------|-------------|-----------|
| **Регрессия**   | `Age`        | MAE         | **0.0821** |
|                 |              | RMSE        | **0.1045** |
| **Классификация** | `Transported` | **ROC-AUC** | **0.792** |

Деревья решений хорошо интерпретируемы и не требуют масштабирования. Регуляризация (`max_depth`, `min_samples_leaf`) предотвращает переобучение.
